# BÀI TOÁN HỒI QUY TUYẾN TÍNH OLS & ƯỚC LƯỢNG PHƯƠNG SAI SAI SỐ
## Phân tích tác động của Số giờ tự học đến Điểm GPA sinh viên

**Mô hình hồi quy tuyến tính đơn:**
$$Y_i = \beta_0 + \beta_1 X_i + u_i, \quad i = 1, \dots, n$$

Trong đó:
- **Biến phụ thuộc ($Y$):** `gpa` – Điểm trung bình tích lũy thang 4.
- **Biến độc lập ($X$):** `gio_tu_hoc_tuan` – Số giờ tự học mỗi tuần.
- **Sai số ngẫu nhiên ($u_i$):** Đại diện cho **các yếu tố không đưa vào mô hình** (năng khiếu, mức độ tập trung, phương pháp học, độ khó ngành học, sức khỏe ngày thi,...). Giả thiết cổ điển: $u_i \sim \mathcal{N}(0, \sigma^2)$ độc lập với $X_i$.

**Nội dung thực hiện:**
1. Áp dụng **Phương pháp Bình phương Cực tiểu (OLS)** để ước lượng các hệ số $\beta_0, \beta_1$.
2. Ước lượng điểm cho **phương sai của các yếu tố nhiễu $\sigma^2$** qua phương sai sai số mẫu $s^2 = \dfrac{\text{SSe}}{n - 2}$.
3. Sử dụng **Phân phối Student $t(n-2)$** để xây dựng khoảng tin cậy 95% và kiểm định ý nghĩa thống kê cho các hệ số $\beta_0, \beta_1$.
4. Sử dụng **Phân phối Khi bình phương $\chi^2(n-2)$** để xây dựng khoảng tin cậy 95% cho phương sai sai số $\sigma^2$.
5. Đối chiếu kết quả ước lượng từ mẫu với các tham số chân lý (Ground Truth) khi sinh dữ liệu.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

# Đọc file dữ liệu
duong_dan = Path("../dataset/sinh_vien.csv")
if not duong_dan.exists():
    duong_dan = Path("dataset/sinh_vien.csv")

df = pd.read_csv(duong_dan)

# Lấy 2 biến X và Y
x = df["gio_tu_hoc_tuan"].to_numpy()
y = df["gpa"].to_numpy()
n = len(df)

print(f"Cỡ mẫu (n): {n}")
print(f"Biến độc lập X (gio_tu_hoc_tuan): min = {x.min():.1f}, max = {x.max():.1f}, trung bình = {x.mean():.4f}")
print(f"Biến phụ thuộc Y (gpa):           min = {y.min():.2f}, max = {y.max():.2f}, trung bình = {y.mean():.4f}")

Cỡ mẫu (n): 200
Biến độc lập X (gio_tu_hoc_tuan): min = 1.0, max = 31.7, trung bình = 15.1060
Biến phụ thuộc Y (gpa):           min = 1.64, max = 4.00, trung bình = 2.9261


## 1. Phương pháp Bình phương Cực tiểu (OLS) tìm $\beta_0, \beta_1$

Đường hồi quy mẫu (SRF) có dạng:
$$\hat{Y}_i = \hat{\beta}_0 + \hat{\beta}_1 X_i$$

Phương pháp OLS tìm nghiệm $(\hat{\beta}_0, \hat{\beta}_1)$ cực tiểu hóa tổng bình phương các sai lệch (phần dư):
$$\min_{\beta_0, \beta_1} \sum_{i=1}^n e_i^2 = \min_{\beta_0, \beta_1} \sum_{i=1}^n (Y_i - \beta_0 - \beta_1 X_i)^2$$

Công thức nghiệm:
$$\hat{\beta}_1 = \frac{S_{xy}}{S_{xx}} = \frac{\sum_{i=1}^n (X_i - \bar{X})(Y_i - \bar{Y})}{\sum_{i=1}^n (X_i - \bar{X})^2}$$
$$\hat{\beta}_0 = \bar{Y} - \hat{\beta}_1 \bar{X}$$

Trong đó $\bar{X} = \dfrac{1}{n}\sum X_i$ và $\bar{Y} = \dfrac{1}{n}\sum Y_i$.

In [ ]:
# 1. Tính các đại lượng thống kê cơ bản
x_bar = np.mean(x)
y_bar = np.mean(y)

S_xx = np.sum((x - x_bar) ** 2)
S_yy = np.sum((y - y_bar) ** 2)
S_xy = np.sum((x - x_bar) * (y - y_bar))

# 2. Ước lượng OLS
beta_1 = S_xy / S_xx
beta_0 = y_bar - beta_1 * x_bar

print("=" * 60)
print("KẾT QUẢ ƯỚC LƯỢNG OLS CÁC HỆ SỐ HỒI QUY:")
print(f"  - Trung bình mẫu X (gio_tu_hoc):  x_bar = {x_bar:.4f}")
print(f"  - Trung bình mẫu Y (gpa):          y_bar = {y_bar:.4f}")
print(f"  - S_xx = {S_xx:.4f}, S_xy = {S_xy:.4f}")
print(f"  -> Hệ số góc (beta_1):             {beta_1:.4f}")
print(f"  -> Hệ số chặn (beta_0):            {beta_0:.4f}")
print(f"  => Phương trình hồi quy mẫu: GPA_hat = {beta_0:.4f} + {beta_1:.4f} * gio_tu_hoc")
print("=" * 60)

KẾT QUẢ ƯỚC LƯỢNG OLS CÁC HỆ SỐ HỒI QUY:
  - Trung bình mẫu X (gio_tu_hoc):  x_bar = 15.1060
  - Trung bình mẫu Y (gpa):          y_bar = 2.9261
  - S_xx = 5740.1928, S_xy = 296.9171
  -> Hệ số góc (beta_1):             0.0517
  -> Hệ số chặn (beta_0):            2.1447
  => Phương trình hồi quy mẫu: GPA_hat = 2.1447 + 0.0517 * gio_tu_hoc


## 2. Ước lượng Phương sai của các yếu tố không đưa vào mô hình ($\sigma^2$)

Phần dư của mỗi quan sát là chênh lệch giữa giá trị thực tế và giá trị dự báo:
$$e_i = Y_i - \hat{Y}_i = Y_i - (\hat{\beta}_0 + \hat{\beta}_1 X_i)$$

Tổng bình phương phần dư (Residual Sum of Squares - $\text{RSS}$):
$$\text{RSS} = \sum_{i=1}^n e_i^2 = S_{yy} - \hat{\beta}_1 S_{xy}$$

Do phải ước lượng 2 tham số ($\beta_0, \beta_1$), bậc tự do của phần dư là $n - 2$.
Ước lượng không chệch cho **phương sai của các yếu tố không đưa vào mô hình $\sigma^2$** là:
$$s^2 = \hat{\sigma}^2 = \frac{\text{RSS}}{n - 2}$$

Và độ lệch chuẩn mẫu của sai số (Standard Error of the Regression - SER):
$$s = \hat{\sigma} = \sqrt{s^2}$$

Hệ số xác định $R^2$ biểu thị tỷ lệ phần trăm sự biến thiên của $Y$ được giải thích bởi $X$:
$$R^2 = 1 - \frac{\text{RSS}}{S_{yy}}$$

In [ ]:
# Giá trị dự báo và phần dư OLS
y_hat = beta_0 + beta_1 * x
residuals = y - y_hat

# Tổng bình phương phần dư (RSS)
rss = np.sum(residuals ** 2)
df_resid = n - 2  # 200 - 2 = 198 bậc tự do

# Ước lượng phương sai và độ lệch chuẩn của sai số ngẫu nhiên
s2 = rss / df_resid
s = np.sqrt(s2)

# Hệ số xác định R^2
r2 = 1 - (rss / S_yy)

print("=" * 60)
print("ƯỚC LƯỢNG PHƯƠNG SAI CÁC YẾU TỐ KHÔNG ĐƯA VÀO MÔ HÌNH (u):")
print(f"  - Tổng bình phương phần dư (RSS): {rss:.4f}")
print(f"  - Bậc tự do (df = n - 2):         {df_resid}")
print(f"  - Phương sai mẫu s^2 (sigma^2):    {s2:.4f}")
print(f"  - Độ lệch chuẩn mẫu s (sigma):     {s:.4f}")
print(f"  - Hệ số xác định R^2:             {r2:.4f} ({r2 * 100:.2f}%)")
print("=" * 60)

ƯỚC LƯỢNG PHƯƠNG SAI CÁC YẾU TỐ KHÔNG ĐƯA VÀO MÔ HÌNH (u):
  - Tổng bình phương phần dư (RSS): 36.8493
  - Bậc tự do (df = n - 2):         198
  - Phương sai mẫu s^2 (sigma^2):    0.1861
  - Độ lệch chuẩn mẫu s (sigma):     0.4314
  - Hệ số xác định R^2:             0.2941 (29.41%)


## 3. Khoảng tin cậy và Kiểm định bằng Phân phối Student $t(n-2)$

Phương sai của các ước lượng hệ số hồi quy:
$$\operatorname{Var}(\hat{\beta}_1) = \frac{\sigma^2}{S_{xx}}, \quad \operatorname{Var}(\hat{\beta}_0) = \sigma^2 \left(\frac{1}{n} + \frac{\bar{X}^2}{S_{xx}}\right)$$

Thay phương sai tổng thể $\sigma^2$ bằng ước lượng mẫu $s^2$, ta có **sai số chuẩn (Standard Error - SE)**:
$$\text{SE}(\hat{\beta}_1) = \frac{s}{\sqrt{S_{xx}}}, \quad \text{SE}(\hat{\beta}_0) = s \sqrt{\frac{1}{n} + \frac{\bar{X}^2}{S_{xx}}}$$

Khi các giả thiết OLS cổ điển thỏa mãn, đại lượng thống kê:
$$T = \frac{\hat{\beta}_j - \beta_j}{\text{SE}(\hat{\beta}_j)} \;\sim\; t(n - 2)$$
tuân theo **phân phối Student** với $n - 2$ bậc tự do.

Do đó, **khoảng tin cậy $1 - \alpha$ cho $\beta_j$** là:
$$\left( \hat{\beta}_j - t_{\alpha/2}(n-2) \cdot \text{SE}(\hat{\beta}_j) \;;\; \hat{\beta}_j + t_{\alpha/2}(n-2) \cdot \text{SE}(\hat{\beta}_j) \right)$$

Đồng thời, ta kiểm định giả thuyết $H_0: \beta_1 = 0$ (số giờ tự học không ảnh hưởng tới GPA) với thống kê:
$$t = \frac{\hat{\beta}_1}{\text{SE}(\hat{\beta}_1)}$$

In [ ]:
alpha = 0.05
df_t = n - 2

# Phân vị Student t_(alpha/2) với n-2 bậc tự do
t_crit = stats.t.ppf(1 - alpha / 2, df=df_t)

# Sai số chuẩn của các hệ số
se_beta1 = s / np.sqrt(S_xx)
se_beta0 = s * np.sqrt(1 / n + (x_bar ** 2) / S_xx)

# Khoảng tin cậy 95%
ci_beta1 = (beta_1 - t_crit * se_beta1, beta_1 + t_crit * se_beta1)
ci_beta0 = (beta_0 - t_crit * se_beta0, beta_0 + t_crit * se_beta0)

# Kiểm định giả thuyết H0: beta_j = 0
t_stat_beta1 = beta_1 / se_beta1
p_val_beta1 = 2 * (1 - stats.t.cdf(np.abs(t_stat_beta1), df=df_t))

t_stat_beta0 = beta_0 / se_beta0
p_val_beta0 = 2 * (1 - stats.t.cdf(np.abs(t_stat_beta0), df=df_t))

print("=" * 60)
print(f"KHOẢNG TIN CẬY 95% DÙNG PHÂN PHỐI STUDENT t({df_t}):")
print(f"  - Phân vị tới hạn t_(0.025)({df_t}): {t_crit:.4f}")
print("-" * 60)
print("1. Hệ số góc (beta_1 - tác động của giờ tự học):")
print(f"   - Sai số chuẩn SE(beta_1) = {se_beta1:.5f}")
print(f"   - Khoảng tin cậy 95%:      [{ci_beta1[0]:.4f} ; {ci_beta1[1]:.4f}]")
print(f"   - Kiểm định H0: beta_1 = 0: t = {t_stat_beta1:.4f}, p-value = {p_val_beta1:.4e}")
print("-" * 60)
print("2. Hệ số chặn (beta_0 - GPA cơ sở):")
print(f"   - Sai số chuẩn SE(beta_0) = {se_beta0:.5f}")
print(f"   - Khoảng tin cậy 95%:      [{ci_beta0[0]:.4f} ; {ci_beta0[1]:.4f}]")
print(f"   - Kiểm định H0: beta_0 = 0: t = {t_stat_beta0:.4f}, p-value = {p_val_beta0:.4e}")
print("=" * 60)

KHOẢNG TIN CẬY 95% DÙNG PHÂN PHỐI STUDENT t(198):
  - Phân vị tới hạn t_(0.025)(198): 1.9720
------------------------------------------------------------
1. Hệ số góc (beta_1 - tác động của giờ tự học):
   - Sai số chuẩn SE(beta_1) = 0.00569
   - Khoảng tin cậy 95%:      [0.0405 ; 0.0630]
   - Kiểm định H0: beta_1 = 0: t = 9.0863, p-value = 1.0569e-16
------------------------------------------------------------
2. Hệ số chặn (beta_0 - GPA cơ sở):
   - Sai số chuẩn SE(beta_0) = 0.09132
   - Khoảng tin cậy 95%:      [1.9647 ; 2.3248]
   - Kiểm định H0: beta_0 = 0: t = 23.4862, p-value = 1.0006e-57


## 5. Bảng tổng hợp & Đối chiếu với Tham số Chân lý (Ground Truth)

Theo file `coding/tao_du_lieu.py`, quy luật sinh dữ liệu thực tế là:
$$\text{GPA} = 2.0 + 0.06 \times \text{gio\_tu\_hoc} + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0;\, 0.4^2)$$

Bảng so sánh giữa **tham số thực tế được gán** và **ước lượng OLS trên mẫu 200 sinh viên**:

| Tham số | Ý nghĩa | Giá trị thực (Ground Truth) | Ước lượng OLS | Khoảng tin cậy 95% | Kết luận |
|---|---|---|---|---|---|
| $\beta_0$ | Hệ số chặn (GPA cơ sở khi giờ học = 0) | $2.0000$ | $2.1447$ | $[1.9647 ; 2.3248]$ | **Chứa giá trị thực $2.0$** |
| $\beta_1$ | Hệ số góc (Ảnh hưởng của 1 giờ học lên GPA) | $0.0600$ | $0.0517$ | $[0.0405 ; 0.0630]$ | **Chứa giá trị thực $0.06$** |
| $\sigma^2$ | Phương sai các yếu tố không đưa vào mô hình | $0.4^2 = 0.1600$ | $s^2 = 0.1861$ | $[0.1543 ; 0.2290]$ | **Chứa giá trị thực $0.16$** |
| $\sigma$ | Độ lệch chuẩn sai số ngẫu nhiên | $0.4000$ | $s = 0.4314$ | $[0.3928 ; 0.4785]$ | **Chứa giá trị thực $0.40$** |

### Nhận xét & Kết luận:
1. **Sự phù hợp của phương pháp:** Toàn bộ các khoảng tin cậy 95% (tính từ phân phối Student cho $\beta$ và Khi bình phương cho $\sigma^2$) đều **bao trùm chính xác giá trị tham số chân lý** ban đầu. Điều này minh chứng tính đúng đắn và hiệu quả của phương pháp OLS trong việc khôi phục các quy luật tiềm ẩn từ dữ liệu mẫu.
2. **Ý nghĩa thực tiễn:** Hệ số $\beta_1 \approx 0.0517 > 0$ có mức ý nghĩa thống kê rất cao ($p < 10^{-15}$). Trung bình, sinh viên tăng 1 giờ tự học mỗi tuần thì GPA tích lũy tăng khoảng $0.05$ điểm (tăng 10 giờ tự học giúp GPA tăng thêm khoảng nửa điểm).
3. **Phương sai sai số:** Ước lượng $s^2 = 0.1861$ phản ánh mức độ biến thiên của GPA gây ra bởi các yếu tố bên ngoài (không quan sát được). Giá trị $s^2$ từ mẫu hơi cao hơn $0.16$ một chút do tác động của việc cắt biên `np.clip(gpa, 1.0, 4.0)` và làm tròn 2 chữ số thập phân khi xuất dữ liệu.